# EcoSupport-Copilot — Data Preparation (01)

Outputs:
- data/kb/passages.jsonl
- data/processed/retriever_train.jsonl
- data/processed/reranker_train.jsonl
- data/processed/generator_train.jsonl
- data/processed/dpo_pairs.jsonl
- data/synthetic_tool_labels/tool_train.jsonl

In [ ]:
import os
import re
import json
import random
from typing import Any, Dict, List, Tuple

from datasets import load_dataset
from transformers import AutoTokenizer
from rank_bm25 import BM25Okapi

random.seed(42)

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath(os.getcwd())
DATA_RAW = os.path.join(ROOT, 'data', 'raw')
DATA_KB = os.path.join(ROOT, 'data', 'kb')
DATA_PROC = os.path.join(ROOT, 'data', 'processed')
DATA_SYN = os.path.join(ROOT, 'data', 'synthetic_tool_labels')

for p in [DATA_RAW, DATA_KB, DATA_PROC, DATA_SYN]:
    os.makedirs(p, exist_ok=True)

def write_jsonl(path: str, rows: List[Dict[str, Any]]):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')

def read_jsonl(path: str) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

def clean_text(text: str) -> str:
    if text is None:
        return ''
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.replace('\u00a0', ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

## 3.1 KB Construction

In [ ]:
# Required download snippet (per spec)
from datasets import load_dataset
bitext = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
msmarco = load_dataset('microsoft/ms_marco', 'v2.1')

# Optional supplement
try:
    faq = load_dataset('Bellerophon/amazon-faq-dataset')
except Exception as e:
    faq = None
    print('Optional FAQ dataset not loaded:', e)

tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
MAX_TOKENS = 256
OVERLAP = 32

def chunk_with_offsets(text: str, max_tokens: int = MAX_TOKENS, overlap: int = OVERLAP) -> List[Tuple[str, int, int]]:
    text = clean_text(text)
    if not text:
        return []
    enc = tokenizer(text, return_offsets_mapping=True, add_special_tokens=False)
    offsets = enc.get('offset_mapping', [])
    if not offsets:
        return []
    chunks: List[Tuple[str, int, int]] = []
    start_tok = 0
    step = max(1, max_tokens - overlap)
    while start_tok < len(offsets):
        end_tok = min(len(offsets), start_tok + max_tokens)
        span_start = int(offsets[start_tok][0])
        span_end = int(offsets[end_tok - 1][1])
        chunk_text = text[span_start:span_end].strip()
        if chunk_text:
            chunks.append((chunk_text, span_start, span_end))
        if end_tok >= len(offsets):
            break
        start_tok += step
    return chunks

HF_SOURCE = 'https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset'

def iter_bitext_rows():
    split = bitext['train'] if 'train' in bitext else list(bitext.values())[0]
    for r in split:
        q = r.get('instruction') or r.get('question') or r.get('user') or r.get('query')
        a = r.get('response') or r.get('answer') or r.get('assistant')
        cat = r.get('category') or r.get('intent') or 'general'
        if q and a:
            yield {'question': str(q), 'answer': str(a), 'category': str(cat)}

seen = set()
passages: List[Dict[str, Any]] = []
idx_by_cat: Dict[str, int] = {}

for row in iter_bitext_rows():
    category = re.sub(r'[^a-z0-9]+', '_', row['category'].lower()).strip('_') or 'general'
    ans = clean_text(row['answer'])
    if not ans:
        continue
    key = ans.lower()
    if key in seen:
        continue
    seen.add(key)
    idx_by_cat.setdefault(category, 0)
    base_id = idx_by_cat[category]
    for j, (chunk_text, span_start, span_end) in enumerate(chunk_with_offsets(ans)):
        doc_id = f'KB_{category}_{base_id}_{j}'
        passages.append({'doc_id': doc_id, 'passage_text': chunk_text, 'category': category, 'span_start': span_start, 'span_end': span_end, 'source_url': HF_SOURCE})
    idx_by_cat[category] += 1

passages_path = os.path.join(DATA_KB, 'passages.jsonl')
write_jsonl(passages_path, passages)
print('KB passages:', len(passages), '->', passages_path)